In [36]:
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import List, Literal


In [37]:
load_dotenv()  

True

In [38]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
ans=model.invoke("What is the capital of France?")
print(ans.content)

The capital of France is **Paris**.


In [39]:
class state(TypedDict):
    content:str
    response:str
  

In [40]:
class subheading(BaseModel): 
    
    title: str = Field(description="Title of the subheading")
    points: List[str] = Field(
        description="Multiple detailed points under this subheading"
    )



In [41]:
class response(subheading):
    tone : Literal["formal", "informal"] = Field(..., description="The tone of the response")
    subheadings: List[subheading] = Field(
            description="Multiple subheadings, each containing its own points"
        )
    

In [42]:
content="Honestly, life becomes a lot more interesting when you start trying new things and stepping out of your comfort zone. Think about it: how many times have you wanted to learn something cool, start a new hobby, or work on an exciting project but ended up putting it off because you thought it would be too difficult? We've all been there! Sometimes, we spend so much time worrying about whether we'll succeed that we forget to actually give things a try. The truth is, you don't have to be an expert to get started, and you definitely don't need to have everything figured out from day one. You just need to take that first step and see where it takes you. Of course, things won't always go as planned. There will be days when nothing seems to work, when you feel stuck, or when you start wondering if you're even making progress. But that's completely normal, and honestly, those moments often teach you the most. Instead of getting frustrated every time something goes wrong, try looking at it as an opportunity to learn something new. Even a tiny mistake can help you understand things better and avoid the same problem next time. Another thing worth remembering is that everyone has their own pace. It's really easy to look at someone else's achievements on social media and feel like you're falling behind. You see people building amazing projects, travelling to incredible places, or achieving their goals, and suddenly your own progress doesn't seem like much. But here's the thing: you're only seeing a small part of their story, not all the hard work, failures, and struggles that happened behind the scenes. So, instead of constantly comparing yourself to others, focus on your own journey and celebrate the little things you accomplish along the way. Maybe you finally understood a concept that confused you for days, finished a project you had been putting off, or simply managed to stay consistent with something you wanted to improve. These things might seem small, but they matter more than you think. It's also important to make time for things you genuinely enjoy. Life shouldn't be all about studying, working, meeting deadlines, and worrying about the future. Sometimes, you just need to hang out with your friends, watch your favourite show, listen to music, go for a walk, or do absolutely nothing for a while. Taking a break doesn't mean you're wasting time; it means you're giving yourself a chance to recharge. At the end of the day, you don't need to have a perfect plan or know exactly where you're headed. Just keep learning, stay curious, try new things, and give yourself some room to make mistakes. Some days will be amazing, while others might feel completely unproductive, and that's okay. What really matters is that you keep moving forward, even if it's just one small step at a time. You never know what opportunities might come your way when you stop overthinking everything and start giving yourself a chance."

In [43]:
model=model.with_structured_output(response)

In [53]:
def gensum(state:state)->state:
    ans=model.invoke(state["content"])
    # state["response"]=ans.model_dump()
    state["response"]=ans
    return state

In [58]:
graph=StateGraph(state)
graph.add_node("summary",gensum)
graph.add_edge(START,"summary")
graph.add_edge("summary",END)
workflow=graph.compile()
workflow.invoke({"content":content})['response'].tone

'informal'